# Scene Detection and Splitting Experiment

Detect hard cuts, inspect compact boundary images and a low-quality labelled review video, choose scenes to skip, persist raw and accepted boundaries, and optionally extract scene clips.

## 1. Configuration

In [ ]:
from pathlib import Path

WORKING_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIRECTORY.parent if WORKING_DIRECTORY.name == "notebooks" else WORKING_DIRECTORY
INPUT_VIDEO = PROJECT_ROOT / "input" / "source.mp4"
SCENE_RUN_NAME = f"{INPUT_VIDEO.stem}-scenes"
SCENE_RUN_DIRECTORY = PROJECT_ROOT / "runs" / SCENE_RUN_NAME

# Global pipeline window. End values are exclusive; use time OR frame per boundary.
DETECTION_START_SECONDS = None
DETECTION_END_SECONDS = None
DETECTION_START_FRAME = None
DETECTION_END_FRAME = None

mode = "ANALYSIS"  # ANALYSIS shows review tools; PRODUCTION detects and saves only.
mode = mode.upper()
if mode not in {"ANALYSIS", "PRODUCTION"}:
    raise ValueError("mode must be 'ANALYSIS' or 'PRODUCTION'")

print(f"Source: {INPUT_VIDEO}")
print(f"Scene run: {SCENE_RUN_DIRECTORY}")
print(f"Mode: {mode}")


## 2. Load project code and inspect the source

In [ ]:
import base64
import sys
from dataclasses import replace

import cv2
import ipywidgets as widgets
import pandas as pd
from IPython.display import HTML, Video, display

SRC_DIRECTORY = PROJECT_ROOT / "src"
if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))
get_ipython().run_line_magic("load_ext", "person_tracker.notebook_magics")

from person_tracker.io import inspect_video, read_nth_frame
from person_tracker.scene_detection import resolve_scene_window

video_info = inspect_video(INPUT_VIDEO)
SCENE_START_FRAME, SCENE_END_FRAME = resolve_scene_window(
    video_info,
    start_seconds=DETECTION_START_SECONDS,
    end_seconds=DETECTION_END_SECONDS,
    start_frame=DETECTION_START_FRAME,
    end_frame=DETECTION_END_FRAME,
)
SCENE_START_SECONDS = SCENE_START_FRAME / video_info.fps
SCENE_END_SECONDS = SCENE_END_FRAME / video_info.fps

print(f"Resolution: {video_info.width} × {video_info.height}")
print(f"Frame rate: {video_info.fps:.6f}")
print(f"Frames: {video_info.frame_count:,}")
print(f"Duration: {video_info.duration_seconds:.2f}s")
print(
    f"Pipeline window: frames {SCENE_START_FRAME:,}:{SCENE_END_FRAME:,} "
    f"({SCENE_START_SECONDS:.3f}s–{SCENE_END_SECONDS:.3f}s)"
)


## 3. Detect and refine automatic scene cuts

In [ ]:
# Sensitive hard-cut detector controls
SCAN_STRIDE = 1
# Ignore boundaries closer together than this. Increase it to suppress brief changes.
MIN_SCENE_SECONDS = 0.35
# Higher correlation and lower difference thresholds detect more subtle cuts.
HISTOGRAM_CORRELATION_THRESHOLD = 0.75
MEAN_DIFFERENCE_THRESHOLD = 12.0
EXTREME_DIFFERENCE_THRESHOLD = 35.0

from tqdm.auto import tqdm
from person_tracker.scene_detection import detect_scene_cuts

progress_state = {"bar": None}
def update_detection_progress(done, total):
    if progress_state["bar"] is None:
        progress_state["bar"] = tqdm(total=total, desc="Detecting scene cuts")
    progress_state["bar"].update(done - progress_state["bar"].n)

video_info, automatic_cuts = detect_scene_cuts(
    INPUT_VIDEO,
    start_frame=SCENE_START_FRAME,
    end_frame=SCENE_END_FRAME,
    scan_stride=SCAN_STRIDE,
    min_scene_seconds=MIN_SCENE_SECONDS,
    histogram_correlation_threshold=HISTOGRAM_CORRELATION_THRESHOLD,
    mean_difference_threshold=MEAN_DIFFERENCE_THRESHOLD,
    extreme_difference_threshold=EXTREME_DIFFERENCE_THRESHOLD,
    progress_callback=update_detection_progress,
)
if progress_state["bar"] is not None:
    progress_state["bar"].close()

print(f"Automatic cuts: {len(automatic_cuts):,}")
for cut in automatic_cuts:
    print(
        f"frame={cut.frame:>7} time={cut.frame/video_info.fps:>9.3f}s "
        f"score={cut.score:>7.2f} correlation={cut.correlation:>7.3f} "
        f"difference={cut.mean_difference:>7.2f}"
    )


## 4. Apply review edits and construct contiguous scenes

The automatic result is preserved separately. Small start, middle, and end images are included directly in the scene table.

In [ ]:
# Manual boundary corrections and initial scene state
MANUAL_CUT_SECONDS = []
REMOVE_CUT_FRAMES = []
DISABLED_SCENE_IDS = []

from person_tracker.scene_detection import build_scene_records, combine_scene_cuts

accepted_cuts = combine_scene_cuts(
    automatic_cuts, fps=video_info.fps, frame_count=video_info.frame_count,
    manual_cut_seconds=MANUAL_CUT_SECONDS, remove_cut_frames=REMOVE_CUT_FRAMES,
)
raw_scenes = build_scene_records(
    video_info, automatic_cuts,
    start_frame=SCENE_START_FRAME, end_frame=SCENE_END_FRAME,
)
scenes = build_scene_records(
    video_info, accepted_cuts, disabled_scene_ids=DISABLED_SCENE_IDS,
    start_frame=SCENE_START_FRAME, end_frame=SCENE_END_FRAME,
)

# Keep this stage fast: thumbnails are created only in the selection stage.
scene_rows = []
for scene in scenes:
    scene_rows.append({
        "Scene": scene.scene_id,
        "Enabled": scene.enabled,
        "Start frame": scene.start_frame,
        "End frame": scene.end_frame,
        "Start": f"{scene.start_seconds:.3f}s",
        "End": f"{scene.end_seconds:.3f}s",
        "Duration": f"{scene.duration_seconds:.3f}s",
        "Boundary": scene.boundary_source,
    })
scene_table = pd.DataFrame(scene_rows)

# pd.set_option("display.max_rows", None)
display(scene_table)
print(f"Raw scenes: {len(raw_scenes):,}")
print(f"Accepted scenes: {len(scenes):,}")


## 5. Create a labelled review video

Create a low-resolution review copy before choosing scenes to skip. The overlay shows the scene number, source-time range, and duration.


In [ ]:
import importlib
import person_tracker.scene_detection as scene_detection

scene_detection = importlib.reload(scene_detection)

create_scene_review_video = scene_detection.create_scene_review_video
create_selected_scene_review_video = (
    scene_detection.create_selected_scene_review_video
)

In [ ]:
%%skip_if_mode PRODUCTION

CREATE_REVIEW_VIDEO = True
REVIEW_VIDEO_WIDTH = 480
REVIEW_VIDEO_QUALITY = 20
REVIEW_VIDEO_DISPLAY_WIDTH = 1080

REVIEW_VIDEO = SCENE_RUN_DIRECTORY / "scene-review.mp4"
CLEAN_REVIEW_VIDEO = SCENE_RUN_DIRECTORY / "scene-review-clean.mp4"

import importlib

from IPython.display import Video, display
from tqdm.auto import tqdm
from person_tracker.io import inspect_video
import person_tracker.scene_detection as scene_detection

scene_detection = importlib.reload(scene_detection)
create_scene_review_video = scene_detection.create_scene_review_video

if CREATE_REVIEW_VIDEO:
    REVIEW_START_FRAME = scenes[0].start_frame
    REVIEW_END_FRAME = scenes[-1].end_frame
    REVIEW_START_SECONDS = scenes[0].start_seconds
    REVIEW_END_SECONDS = scenes[-1].end_seconds

    review_duration = REVIEW_END_SECONDS - REVIEW_START_SECONDS
    expected_frame_count = REVIEW_END_FRAME - REVIEW_START_FRAME

    review_progress = tqdm(
        total=review_duration,
        desc="Creating review videos",
        unit="s",
    )

    def update_review_progress(processed_seconds, total_seconds):
        review_progress.total = total_seconds
        review_progress.n = min(
            processed_seconds,
            total_seconds,
        )
        review_progress.refresh()

    try:
        review_path = create_scene_review_video(
            INPUT_VIDEO,
            scenes,
            REVIEW_VIDEO,
            clean_output_path=CLEAN_REVIEW_VIDEO,
            width=REVIEW_VIDEO_WIDTH,
            quality=REVIEW_VIDEO_QUALITY,
            progress_callback=update_review_progress,
        )
    finally:
        review_progress.close()

    review_info = inspect_video(REVIEW_VIDEO)
    clean_review_info = inspect_video(CLEAN_REVIEW_VIDEO)

    display(Video(
        filename=str(review_path),
        embed=True,
        width=REVIEW_VIDEO_DISPLAY_WIDTH,
    ))

    print(f"Saved labelled review video: {REVIEW_VIDEO}")
    print(f"Saved clean review video: {CLEAN_REVIEW_VIDEO}")
    print(
        f"Source window: frames "
        f"{REVIEW_START_FRAME:,}–{REVIEW_END_FRAME - 1:,}"
    )
    print(
        f"Labelled review: {review_info.frame_count:,} frames, "
        f"{review_info.duration_seconds:.3f}s"
    )
    print(
        f"Clean review: {clean_review_info.frame_count:,} frames, "
        f"{clean_review_info.duration_seconds:.3f}s"
    )
    print(f"Expected approximately: {expected_frame_count:,} frames")
else:
    print("Review-video creation disabled.")

## 6. Choose scenes to keep or skip

After watching the labelled review video, uncheck any scenes that should not continue to later processing stages.


In [ ]:
%%skip_if_mode PRODUCTION

import hashlib
import json
import subprocess
import time

import ipywidgets as widgets

from dataclasses import replace
from pathlib import Path
from IPython.display import display
from person_tracker.io import inspect_video

SELECTION_IMAGE_WIDTH = 140
SELECTION_IMAGE_GAP = 8
SELECTION_JPEG_QUALITY = 60
SCENE_CARD_WIDTH = 470

REVIEW_START_FRAME = scenes[0].start_frame
REVIEW_END_FRAME = scenes[-1].end_frame
EXPECTED_REVIEW_FRAME_COUNT = REVIEW_END_FRAME - REVIEW_START_FRAME

In [ ]:
def create_scene_thumbnail_sheets(
    review_video,
    scenes,
    thumbnail_root,
    review_start_frame,
    image_width=140,
    image_gap=8,
    jpeg_quality=60,
):
    review_video = Path(review_video).expanduser().resolve()
    thumbnail_root = Path(thumbnail_root).expanduser().resolve()

    if not review_video.exists():
        raise FileNotFoundError(
            f"Clean review video does not exist: {review_video}"
        )

    review_stat = review_video.stat()
    cache_data = {
        "review_video": str(review_video),
        "review_size": review_stat.st_size,
        "review_mtime_ns": review_stat.st_mtime_ns,
        "review_start_frame": review_start_frame,
        "image_width": image_width,
        "image_gap": image_gap,
        "jpeg_quality": jpeg_quality,
        "scenes": [
            [
                scene.scene_id,
                scene.start_frame,
                scene.start_frame + scene.frame_count // 2,
                scene.end_frame - 1,
            ]
            for scene in scenes
        ],
    }
    cache_signature = hashlib.sha256(
        json.dumps(
            cache_data,
            sort_keys=True,
        ).encode("utf-8")
    ).hexdigest()[:16]

    output_directory = thumbnail_root / cache_signature
    output_directory.mkdir(parents=True, exist_ok=True)

    thumbnail_paths = {
        scene.scene_id:
            output_directory / f"thumbnail-{position + 1:06d}.jpg"
        for position, scene in enumerate(scenes)
    }

    if all(path.exists() for path in thumbnail_paths.values()):
        return thumbnail_paths, True

    expected_review_frame_count = (
        scenes[-1].end_frame - review_start_frame
    )
    selected_review_frames = []

    for scene in scenes:
        source_frames = [
            scene.start_frame,
            scene.start_frame + scene.frame_count // 2,
            scene.end_frame - 1,
        ]
        review_frames = [
            frame_number - review_start_frame
            for frame_number in source_frames
        ]

        if len(set(review_frames)) != 3:
            raise ValueError(
                f"{scene.scene_id} does not contain three distinct frames"
            )

        if (
            review_frames[0] < 0
            or review_frames[-1] >= expected_review_frame_count
        ):
            raise ValueError(
                f"{scene.scene_id} has frames outside the review-video window"
            )

        selected_review_frames.extend(review_frames)

    select_expression = "+".join(
        f"eq(n\\,{frame_number})"
        for frame_number in selected_review_frames
    )
    jpeg_qscale = round(
        31 - max(0, min(100, jpeg_quality)) * 29 / 100
    )
    jpeg_qscale = max(2, min(31, jpeg_qscale))

    video_filter = (
        f"select={select_expression},"
        f"scale={image_width}:-2,"
        f"tile=3x1:nb_frames=3:"
        f"padding={image_gap}:margin=0:color=white"
    )
    output_pattern = output_directory / "thumbnail-%06d.jpg"

    command = [
        "ffmpeg",
        "-y",
        "-loglevel", "error",
        "-i", str(review_video),
        "-an",
        "-sn",
        "-dn",
        "-vf", video_filter,
        "-frames:v", str(len(scenes)),
        "-fps_mode", "passthrough",
        "-q:v", str(jpeg_qscale),
        "-start_number", "1",
        str(output_pattern),
    ]

    completed = subprocess.run(
        command,
        capture_output=True,
        text=True,
    )

    if completed.returncode != 0:
        raise RuntimeError(
            f"Could not create scene thumbnails: "
            f"{completed.stderr.strip()}"
        )

    missing_paths = [
        path
        for path in thumbnail_paths.values()
        if not path.exists()
    ]

    if missing_paths:
        raise RuntimeError(
            f"FFmpeg did not create {len(missing_paths)} expected thumbnails"
        )

    (output_directory / "cache.json").write_text(
        json.dumps(
            cache_data,
            indent=2,
        ),
        encoding="utf-8",
    )

    return thumbnail_paths, False

clean_review_info = inspect_video(CLEAN_REVIEW_VIDEO)
review_frame_difference = abs(
    clean_review_info.frame_count - EXPECTED_REVIEW_FRAME_COUNT
)

if review_frame_difference > 2:
    raise RuntimeError(
        f"The clean review video contains "
        f"{clean_review_info.frame_count:,} frames, but the current scene "
        f"window expects approximately {EXPECTED_REVIEW_FRAME_COUNT:,}. "
        f"Rerun the review-video cell."
    )

thumbnail_start_time = time.perf_counter()

scene_thumbnail_paths, used_thumbnail_cache = (
    create_scene_thumbnail_sheets(
        CLEAN_REVIEW_VIDEO,
        scenes,
        SCENE_RUN_DIRECTORY / "thumbnails",
        REVIEW_START_FRAME,
        image_width=SELECTION_IMAGE_WIDTH,
        image_gap=SELECTION_IMAGE_GAP,
        jpeg_quality=SELECTION_JPEG_QUALITY,
    )
)

thumbnail_elapsed = time.perf_counter() - thumbnail_start_time

print(
    f"{'Loaded' if used_thumbnail_cache else 'Created'} "
    f"{len(scene_thumbnail_paths)} scene thumbnails "
    f"in {thumbnail_elapsed:.2f}s"
)

In [ ]:
from ipyevents import Event

scene_checkboxes = {}
scene_cards = []
scene_click_events = []
scene_click_handlers = []
selection_status = widgets.HTML()
scenes = list(scenes)

def update_selection_status():
    enabled_count = sum(
        checkbox.value
        for checkbox in scene_checkboxes.values()
    )
    selection_status.value = (
        f"<b>{enabled_count} of {len(scene_checkboxes)} scenes enabled</b>"
    )

def synchronize_scene_selection(change, scene_position):
    scenes[scene_position] = replace(
        scenes[scene_position],
        enabled=change["new"],
    )
    update_selection_status()

for scene_position, scene in enumerate(scenes):
    start_frame = scene.start_frame
    middle_frame = scene.start_frame + scene.frame_count // 2
    end_frame = scene.end_frame - 1

    checkbox = widgets.Checkbox(
        value=scene.enabled,
        description=(
            f"{scene.scene_id} | "
            f"{scene.start_seconds:.2f}s–{scene.end_seconds:.2f}s | "
            f"{scene.duration_seconds:.2f}s"
        ),
        indent=False,
        layout=widgets.Layout(width="100%"),
    )

    thumbnail = widgets.Image(
        value=scene_thumbnail_paths[scene.scene_id].read_bytes(),
        format="jpeg",
    )

    frame_labels = widgets.HTML(
        f"""
        <div style="
            display:grid;
            grid-template-columns:repeat(3,{SELECTION_IMAGE_WIDTH}px);
            gap:{SELECTION_IMAGE_GAP}px;
            text-align:left;
        ">
            <small>frame {start_frame:,}</small>
            <small>frame {middle_frame:,}</small>
            <small>frame {end_frame:,}</small>
        </div>
        """
    )

    card = widgets.VBox(
        [checkbox, thumbnail, frame_labels],
        layout=widgets.Layout(
            border="1px solid #bbb",
            padding="8px",
            width=f"{SCENE_CARD_WIDTH}px",
            overflow="hidden",
        ),
    )

    def handle_card_click(event, target_checkbox=checkbox):
        target_checkbox.value = not target_checkbox.value

    click_event = Event(
        source=card,
        watched_events=["click"],
        prevent_default_action=True,
    )

    click_event.on_dom_event(handle_card_click)

    checkbox.observe(
        lambda change, position=scene_position:
            synchronize_scene_selection(change, position),
        names="value",
    )

    scene_checkboxes[scene.scene_id] = checkbox
    scene_click_events.append(click_event)
    scene_click_handlers.append(handle_card_click)
    scene_cards.append(card)

update_selection_status()

scene_container = widgets.GridBox(
    scene_cards,
    layout=widgets.Layout(
        grid_template_columns=(
            f"repeat(auto-fit, {SCENE_CARD_WIDTH}px)"
        ),
        grid_gap="12px",
        align_items="flex-start",
        justify_content="flex-start",
    ),
)

display(
    selection_status,
    scene_container,
)

## 7. Create a labelled video from selected scenes

Concatenate only the enabled scenes and create a review video with the same labels and appearance as step 5.


In [ ]:
%%skip_if_mode PRODUCTION

CREATE_SELECTED_REVIEW_VIDEO = True
SELECTED_REVIEW_VIDEO_QUALITY = 20
SELECTED_REVIEW_VIDEO_DISPLAY_WIDTH = 1080
SELECTED_REVIEW_VIDEO = (
    SCENE_RUN_DIRECTORY / "selected-scene-review.mp4"
)

import importlib

from IPython.display import Video, display
from tqdm.auto import tqdm
from person_tracker.io import inspect_video
import person_tracker.scene_detection as scene_detection

scene_detection = importlib.reload(scene_detection)
create_selected_scene_review_video = (
    scene_detection.create_selected_scene_review_video
)

selected_scenes = [
    scene
    for scene in scenes
    if scene.enabled
]

if CREATE_SELECTED_REVIEW_VIDEO and selected_scenes:
    selected_frame_count = sum(
        scene.frame_count
        for scene in selected_scenes
    )
    selected_duration = (
        selected_frame_count / scenes[0].fps
    )

    selected_review_progress = tqdm(
        total=selected_duration,
        desc="Creating selected review video",
        unit="s",
    )

    def update_selected_review_progress(
        processed_seconds,
        total_seconds,
    ):
        selected_review_progress.total = total_seconds
        selected_review_progress.n = min(
            processed_seconds,
            total_seconds,
        )
        selected_review_progress.refresh()

    try:
        selected_review_path = create_selected_scene_review_video(
            REVIEW_VIDEO,
            scenes,
            SELECTED_REVIEW_VIDEO,
            review_start_frame=REVIEW_START_FRAME,
            quality=SELECTED_REVIEW_VIDEO_QUALITY,
            progress_callback=update_selected_review_progress,
        )
    finally:
        selected_review_progress.close()

    selected_review_info = inspect_video(
        selected_review_path
    )

    display(Video(
        filename=str(selected_review_path),
        embed=True,
        width=SELECTED_REVIEW_VIDEO_DISPLAY_WIDTH,
    ))

    print(
        f"Selected scenes: "
        f"{len(selected_scenes)} of {len(scenes)}"
    )
    print(
        f"Selected review: "
        f"{selected_review_info.frame_count:,} frames, "
        f"{selected_review_info.duration_seconds:.3f}s"
    )
    print(
        f"Saved selected-scene review video: "
        f"{selected_review_path}"
    )
elif not selected_scenes:
    print("No scenes are selected.")
else:
    print("Selected-scene review-video creation disabled.")

## 8. Persist the scene run


In [ ]:
from pathlib import Path

from person_tracker.scene_storage import build_scene_manifest, save_scene_run
from person_tracker.storage import update_latest_run

scene_settings = {
    "detector": "histogram_pixel_difference_v1",
    "detection_start_seconds": DETECTION_START_SECONDS,
    "detection_end_seconds": DETECTION_END_SECONDS,
    "detection_start_frame": DETECTION_START_FRAME,
    "detection_end_frame": DETECTION_END_FRAME,
    "effective_start_frame": SCENE_START_FRAME,
    "effective_end_frame": SCENE_END_FRAME,
    "effective_start_seconds": SCENE_START_SECONDS,
    "effective_end_seconds": SCENE_END_SECONDS,
    "scan_stride": SCAN_STRIDE,
    "min_scene_seconds": MIN_SCENE_SECONDS,
    "histogram_correlation_threshold": HISTOGRAM_CORRELATION_THRESHOLD,
    "mean_difference_threshold": MEAN_DIFFERENCE_THRESHOLD,
    "extreme_difference_threshold": EXTREME_DIFFERENCE_THRESHOLD,
    "manual_cut_seconds": list(MANUAL_CUT_SECONDS),
    "remove_cut_frames": list(REMOVE_CUT_FRAMES),
    "disabled_scene_ids": [
        scene.scene_id
        for scene in scenes
        if not scene.enabled
    ],
}

if "REVIEW_VIDEO_WIDTH" in globals():
    scene_settings["review_video_width"] = REVIEW_VIDEO_WIDTH

if "REVIEW_VIDEO_QUALITY" in globals():
    scene_settings["review_video_quality"] = REVIEW_VIDEO_QUALITY

if "SELECTED_REVIEW_VIDEO_QUALITY" in globals():
    scene_settings["selected_review_video_quality"] = (
        SELECTED_REVIEW_VIDEO_QUALITY
    )

scene_manifest = build_scene_manifest(
    video_info,
    settings=scene_settings,
)

scene_manifest["window_start_frame"] = SCENE_START_FRAME
scene_manifest["window_end_frame"] = SCENE_END_FRAME
scene_manifest["window_start_seconds"] = SCENE_START_SECONDS
scene_manifest["window_end_seconds"] = SCENE_END_SECONDS
scene_manifest["artifacts"] = {}

def add_scene_artifact(name, path):
    if path is None:
        return

    path = Path(path).expanduser().resolve()

    if not path.exists():
        return

    try:
        stored_path = path.relative_to(
            SCENE_RUN_DIRECTORY.resolve()
        )
    except ValueError:
        stored_path = path

    scene_manifest["artifacts"][name] = str(stored_path)

add_scene_artifact(
    "labelled_review_video",
    globals().get("REVIEW_VIDEO"),
)
add_scene_artifact(
    "clean_review_video",
    globals().get("CLEAN_REVIEW_VIDEO"),
)
add_scene_artifact(
    "selected_review_video",
    globals().get("SELECTED_REVIEW_VIDEO"),
)

if "scene_thumbnail_paths" in globals() and scene_thumbnail_paths:
    thumbnail_directory = next(
        iter(scene_thumbnail_paths.values())
    ).parent
    add_scene_artifact(
        "thumbnail_directory",
        thumbnail_directory,
    )

save_scene_run(
    SCENE_RUN_DIRECTORY,
    manifest=scene_manifest,
    raw_scenes=raw_scenes,
    scenes=scenes,
)

latest_path = update_latest_run(
    PROJECT_ROOT / "runs",
    SCENE_RUN_DIRECTORY,
    stage="scene",
)

print(
    f"Enabled scenes: "
    f"{sum(scene.enabled for scene in scenes)} of {len(scenes)}"
)
print(
    f"Saved manifest: "
    f"{SCENE_RUN_DIRECTORY / 'manifest.json'}"
)
print(
    f"Saved raw scenes: "
    f"{SCENE_RUN_DIRECTORY / 'raw_scenes.jsonl'}"
)
print(
    f"Saved accepted scenes: "
    f"{SCENE_RUN_DIRECTORY / 'scenes.jsonl'}"
)
print(f"Updated latest scene run: {latest_path}")

if scene_manifest["artifacts"]:
    print("Saved artifacts:")

    for artifact_name, artifact_path in scene_manifest["artifacts"].items():
        print(f"  {artifact_name}: {artifact_path}")

## 10. Reload and verify the saved contract


In [ ]:
from person_tracker.scene_storage import load_scene_run

saved_manifest, saved_raw_scenes, saved_scenes = load_scene_run(SCENE_RUN_DIRECTORY)
assert len(saved_raw_scenes) == len(raw_scenes)
assert len(saved_scenes) == len(scenes)
assert saved_scenes[0].start_frame == SCENE_START_FRAME
assert saved_scenes[-1].end_frame == SCENE_END_FRAME
assert [scene.enabled for scene in saved_scenes] == [scene.enabled for scene in scenes]
print("Scene run reloaded and validated successfully.")
print(
    f"Saved window: frames {saved_manifest['window_start_frame']:,}:"
    f"{saved_manifest['window_end_frame']:,}"
)
print(f"Schema version: {saved_manifest['schema_version']}")


## 11. Create the final full-resolution video

Create a high-quality, full-resolution video containing only the enabled scenes. This processes the original source video in a single bounded-memory pass, preserves its original resolution and frame rate, and produces the final deliverable without embedding the large video in the notebook.

In [ ]:
CREATE_FINAL_SELECTED_VIDEO = True

FINAL_VIDEO_ENCODER = "auto"
FINAL_VIDEO_CRF = 14
FINAL_VIDEO_CPU_PRESET = "medium"
FINAL_VIDEO_NVENC_CQ = 14
FINAL_VIDEO_NVENC_PRESET = "p7"
FINAL_VIDEO_AUDIO_BITRATE = "320k"
FINAL_VIDEO_THREADS = 0

FINAL_SELECTED_VIDEO = (
    SCENE_RUN_DIRECTORY / "selected-scenes-full-quality.mp4"
)

import importlib

from IPython.display import FileLink, display
from tqdm.auto import tqdm
from person_tracker.io import inspect_video
import person_tracker.scene_detection as scene_detection

scene_detection = importlib.reload(scene_detection)

create_selected_scene_video = (
    scene_detection.create_selected_scene_video
)
nvenc_available = scene_detection.nvenc_available

selected_scenes = [
    scene
    for scene in scenes
    if scene.enabled
]

if FINAL_VIDEO_ENCODER == "auto":
    final_video_encoder = (
        "nvenc"
        if nvenc_available(INPUT_VIDEO)
        else "x264"
    )
else:
    final_video_encoder = FINAL_VIDEO_ENCODER

print(f"Final video encoder: {final_video_encoder}")

if CREATE_FINAL_SELECTED_VIDEO and selected_scenes:
    selected_frame_count = sum(
        scene.frame_count
        for scene in selected_scenes
    )
    selected_duration = (
        selected_frame_count / scenes[0].fps
    )

    final_video_progress = tqdm(
        total=selected_duration,
        desc=f"Creating final video ({final_video_encoder})",
        unit="s",
    )

    def update_final_video_progress(
        processed_seconds,
        total_seconds,
    ):
        final_video_progress.total = total_seconds
        final_video_progress.n = min(
            processed_seconds,
            total_seconds,
        )
        final_video_progress.refresh()

    try:
        final_video_path = create_selected_scene_video(
            INPUT_VIDEO,
            scenes,
            FINAL_SELECTED_VIDEO,
            encoder=final_video_encoder,
            crf=FINAL_VIDEO_CRF,
            cpu_preset=FINAL_VIDEO_CPU_PRESET,
            nvenc_cq=FINAL_VIDEO_NVENC_CQ,
            nvenc_preset=FINAL_VIDEO_NVENC_PRESET,
            audio_bitrate=FINAL_VIDEO_AUDIO_BITRATE,
            threads=FINAL_VIDEO_THREADS,
            progress_callback=update_final_video_progress,
        )
    finally:
        final_video_progress.close()

    final_video_info = inspect_video(final_video_path)

    print(
        f"Selected scenes: "
        f"{len(selected_scenes)} of {len(scenes)}"
    )
    print(
        f"Final video: "
        f"{final_video_info.width}×{final_video_info.height}, "
        f"{final_video_info.frame_count:,} frames, "
        f"{final_video_info.duration_seconds:.3f}s"
    )
    print(f"Encoder: {final_video_encoder}")

    if final_video_encoder == "nvenc":
        print(
            f"NVENC preset: {FINAL_VIDEO_NVENC_PRESET}, "
            f"CQ: {FINAL_VIDEO_NVENC_CQ}"
        )
    else:
        print(
            f"x264 preset: {FINAL_VIDEO_CPU_PRESET}, "
            f"CRF: {FINAL_VIDEO_CRF}"
        )

    print(f"Saved final video: {final_video_path}")
    display(FileLink(str(final_video_path)))
elif not selected_scenes:
    print("No scenes are selected.")
else:
    print("Final selected-video creation disabled.")